# Análise Exploratória — Home Credit

Este notebook analisa `Dados/clean_data.csv` antes da construção da ABT. O objetivo é entender qualidade dos dados, desbalanceamento do `TARGET`, perfil financeiro/demográfico e relações preliminares que orientam o uso das variáveis no modelo.

As regras e funções reutilizáveis ficam centralizadas em `DataPipeline/pipeline_functions.py`; o notebook apenas chama essas funções e apresenta os resultados.


### Bloco 1 — Preparação do ambiente
**O que este bloco faz:** localiza a raiz do projeto, adiciona a raiz ao `sys.path`, importa bibliotecas de análise e configura a exibição do Pandas.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name in {"DataPipeline", "Model"}:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from DataPipeline import config
from DataPipeline import pipeline_functions as pf


### Bloco 2 — Carregamento da base limpa
**O que este bloco faz:** lê `Dados/clean_data.csv`, mostra a quantidade de linhas/colunas, valida a unicidade de `SK_ID_CURR` e calcula a taxa geral de inadimplência.

In [ ]:
df = pf.load_clean_data()
overview = pf.dataset_overview(df)

print(f"Linhas: {overview['rows']:,}")
print(f"Colunas: {overview['columns']:,}")
print(f"IDs únicos: {overview['unique_ids']:,}")
print(f"Duplicidades de SK_ID_CURR: {overview['duplicate_ids']:,}")
print(f"Taxa de default: {overview['target_rate']:.2%}")


### Bloco 3 — Distribuição do TARGET
**O que este bloco faz:** conta adimplentes (`TARGET=0`) e inadimplentes (`TARGET=1`), calcula a participação percentual de cada classe e exibe a distribuição em gráfico.

In [ ]:
target_table = pf.target_distribution(df)
display(target_table.style.format({"percentual": "{:.2%}"}))
pf.plot_target_distribution(target_table)


### Bloco 4 — Valores ausentes
**O que este bloco faz:** calcula o percentual de nulos de todas as colunas, ordena da mais vazia para a menos vazia e identifica quantas ultrapassam 50% de ausência.

In [ ]:
missing = pf.missing_summary(df)
display(missing.head(35))
print("Colunas com >50% de nulos:", int((missing["missing_pct"] > 50).sum()))


### Bloco 5 — Estatísticas financeiras e demográficas
**O que este bloco faz:** resume renda, crédito, anuidade, valor do bem, idade em dias, tempo de emprego e scores externos usando média, desvio e percentis.

In [ ]:
display(pf.describe_available_columns(df, config.EDA_PROFILE_COLUMNS))


### Bloco 6 — Idade e inadimplência
**O que este bloco faz:** converte `DAYS_BIRTH` em idade aproximada em anos, cria faixas etárias e calcula a taxa de inadimplência de cada faixa.

In [ ]:
age_summary = pf.age_target_summary(df)
display(age_summary)
pf.plot_age_target_summary(age_summary)


### Bloco 7 — Razões financeiras
**O que este bloco faz:** cria `CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO` e `ANNUITY_CREDIT_RATIO` e compara a mediana dessas razões entre adimplentes e inadimplentes.

In [ ]:
analysis = pf.add_financial_ratios(df)
display(pf.target_group_summary(analysis, config.RATIO_COLUMNS))


### Bloco 8 — Scores externos
**O que este bloco faz:** compara média, mediana e quantidade válida dos `EXT_SOURCE_*` por classe de `TARGET` e mostra a distribuição dos scores para bons e maus pagadores.

In [ ]:
display(pf.external_source_summary(analysis))
pf.plot_density_by_target(analysis, config.EXT_SOURCE_COLUMNS)


### Bloco 9 — Variáveis categóricas
**O que este bloco faz:** para cada categórica prioritária, mostra as categorias mais frequentes, quantidade de solicitações e taxa de inadimplência.

In [ ]:
for col in config.EDA_CATEGORICAL_COLUMNS:
    summary = pf.categorical_target_summary(analysis, col)
    if not summary.empty:
        print()
        print(f"### {col}")
        display(summary.style.format({"default_rate": "{:.2%}"}))


### Bloco 10 — Leitura das fontes auxiliares
**O que este bloco faz:** lê `clean_bureau.csv` e `clean_previous_application.csv` para validar volume, quantidade de clientes cobertos e granularidade antes da agregação.

In [ ]:
bureau, previous = pf.load_clean_auxiliary_sources()
display(pf.auxiliary_sources_summary(bureau, previous))


### Bloco 11 — Conclusões da EDA
**O que este bloco faz:** não executa transformação. Este bloco registra os pontos que devem ser lembrados antes de construir a ABT.

- O `TARGET` é desbalanceado; a avaliação deve olhar AUC-ROC, KS, recall e precision.
- `EXT_SOURCE_*`, renda, crédito e razões financeiras são variáveis relevantes para investigação.
- Colunas extremamente vazias serão removidas na construção da ABT, preservando os scores externos.
- `bureau` e `previous_application` precisam ser agregados por `SK_ID_CURR` antes do join.
- Imputação e encoding serão executados dentro do Pipeline de treinamento.